# Estimating the site frequency spectrum (SFS) from low depth NGS data

The **site frequency spectrum** is one of the most used summaries of genetic variation. For a
sample of $n$ diploid individuals you go through the sites in the genome, count how many
**derived** alleles the sample carries at each site, and tabulate how often each count occurs.
The result is a vector

$$\theta=(\theta_0,\theta_1,\dots,\theta_{2n})$$

where $\theta_z$ is the proportion of sites with exactly $z$ derived alleles out of the $2n$
alleles in the sample. Because it is a proportion, $\sum_{z=0}^{2n}\theta_z = 1$.

Almost everything we estimate from allele frequencies is a function of the SFS: nucleotide
diversity, Watterson's $\theta$, Tajima's D, $F_{ST}$, demographic history and tests for
selection. So if we can estimate the SFS well, we can estimate all of those well.

**The problem.** With low depth NGS data we do not know the genotypes, so we do not know the
allele counts either. The obvious thing to do - call the genotypes and count - turns out to be
badly biased. In this exercise we will see that bias with our own eyes, and then fix it with a
method that never calls a genotype: it works directly on the genotype likelihoods and uses an
EM algorithm, exactly the way `ANGSD`/`realSFS` does it.

**Learning objectives**

 - Know what the SFS is and why it has $2n+1$ categories
 - See why calling genotypes at low depth gives a biased SFS
 - Understand the sample allele frequency likelihood (SAF), $p(X_j\mid Z_j=z)$
 - Be able to write up the likelihood of the SFS and the E- and M-step of the EM algorithm
 - Estimate the SFS from genotype likelihoods and compare it to the truth

Everything below is simulated, which is the nice thing about this exercise: we know the true
answer and can check how close each method gets.

# 1. Simulating the data

We build the data in four steps, each of which mimics one step of a real study

 1. a true allele frequency $f_j$ for each site in the population
 2. true genotypes $g_{ij}$ for $n$ individuals, drawn under Hardy-Weinberg
 3. sequencing reads for each individual with a given depth and error rate
 4. genotype likelihoods computed from those reads - this is what we would get from real data

We then throw away everything except the genotype likelihoods (step 4), estimate the SFS from
them, and compare with the SFS we can calculate from the true genotypes (step 2).

## The population allele frequencies

The frequencies of the sites in a genome are not uniform: most variable sites are rare. A
convenient distribution with that shape is the beta distribution, and we use
$f_j \sim \text{Beta}(0.5, 2)$. This plays the role that the true demographic history would
play for real data - it is what generates the shape of the SFS.

In [ ]:
curve(dbeta(x,0.5, 2),from=0,to=1,main="Beta distribution for population allele frequency",
ylab="density",xlab="Allele frequency (x)")

 - Look at the curve. Are low or high frequencies the most common?
 - Under the standard neutral model the expected SFS is proportional to $1/z$. Does the beta
   distribution above give something that has roughly that shape?

In [ ]:
# Simulate allele frequencies.
#
# Input is a number of sites, output is a vector of allele frequencies.
sim_af <- function(m)
    rbeta(m, 0.5, 2)

palette(c("goldenrod","purple","blue"))

# Simulate
set.seed(1)
m <- 100000
n <- 10

## simulate population allele frequencies
af <- sim_af(m)
cat("/n simulated true frequencies for the first 6 SNPs\n")
head(af)

hist(af,xlab="population allele frequency",col="orange",
  main="Histogram of simulated allele frequencies",
  ylab="Number of sites")

The histogram is the distribution of the *population* frequencies. It is not yet the SFS: the
SFS is about the number of derived alleles in our **sample** of $n$ individuals.

## From frequencies to genotypes

At a site with allele frequency $f_j$ we draw the genotype of each individual under
Hardy-Weinberg equilibrium

$$g_{ij}\sim \text{Binomial}(2, f_j)$$

so $g_{ij}\in\{0,1,2\}$ is the number of derived alleles individual $i$ carries at site $j$.
The number of derived alleles in the whole sample at site $j$ is then

$$Z_j=\sum_{i=1}^{n} g_{ij} \in \{0,1,\dots,2n\}$$

In [ ]:
# Simulate genotypes from allele frequencies.
#
# Input is a number of individuals, and a vector of m alleles frequencies,
# output is a n-by-m matrix.
sim_gt <- function(af, n){
   gt <- sapply(af, rbinom, n = n, size = 2)
  rownames(gt) <- paste0("Ind",1:n)
  colnames(gt) <- paste0("SNP",1:length(af))
  gt
}


## simulate genotypes for n indiviudals based on the frequency
gt <- sim_gt(af, n)
cat("Genotypes for the first 6 simulated SNPs")
gt[,1:6]


cat("\n\nNumber of derived alleles for the first 6 simulated SNPs ")
t(colSums(gt[,1:6]))

cat("\n\nNumber of SNPs with a certain sample allele frequency ")
table(colSums(gt)/2/n)

 - `table(colSums(gt)/2/n)` counts how many sites have each sample frequency. How many
   different categories can there be with $n=10$ individuals?
 - Some sites have zero derived alleles even though the population frequency was above zero.
   Why?

## The SFS from the true genotypes

The SFS is now just a table of $Z_j$ divided by the number of sites

In [ ]:
# Calculate SFS from genotypes.
#
# Input is an n-by-m matrix of genotypes, output is an SFS.
calc_sfs <- function(gt) {
  n <- nrow(gt)
  m <- ncol(gt)
  table(colSums(gt) / (2 * n)) / m
}

true_sfs <- calc_sfs(gt)
barplot(true_sfs,xlab="Number of derived alleles (z)",ylab="fraction of sites (theta)",col=1,main="SFS from true genotypes",names=0:(2*n))

This is the SFS we are trying to recover. Keep the shape in mind - it is the yardstick for
everything below.

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/sfs_quiz1.json")


# 2. Simulating NGS data

Now we pretend we sequenced these individuals. For each individual and site

 - the depth is drawn from a Poisson distribution, $D_{ij}\sim \text{Poisson}(\bar{D}_i)$,
   where $\bar D_i$ is the average depth of the individual
 - each of the $D_{ij}$ reads samples one of the individual's two alleles at random
 - with probability $\epsilon$ the read shows the *other* allele instead (a sequencing error)

To keep it simple we pretend there are only two possible bases instead of the usual four. The
probability that a single read is derived is then

| genotype $g$ | $p(\text{read is derived}\mid g)$ |
|---|---|
| 0 | $\epsilon$ |
| 1 | $0.5$ |
| 2 | $1-\epsilon$ |

and the **genotype likelihood** is the binomial probability of seeing the observed number of
derived reads out of $D_{ij}$ reads

$$p(X_{ij}\mid G_{ij}=g) = \binom{D_{ij}}{k_{ij}}\,p_g^{\,k_{ij}}(1-p_g)^{D_{ij}-k_{ij}}$$

where $k_{ij}$ is the number of reads carrying the derived allele. Note that the genotype
likelihood is **not** a probability of the genotype - it is the probability of the *data* for
each of the three possible genotypes.

In [ ]:
# Simulate genotype likelihoods from genotypes.
#
# Input is a n-by-m matrix of genotypes, output is a n-list length of
# 3-by-m matrices.
sim_gl <- function(gt, error = 0.01, mean_depth = 2, min_depth = 0) {
  n <- nrow(gt)
  m <- ncol(gt)
  e <- c(error, 0.5, 1 - error)
  depths <- pmax(rpois(n * m, mean_depth), min_depth)
  probs <- e[gt + 1]
  alts <- rbinom(n * m, depths, probs)
  gl <- sapply(e, dbinom, x = alts, size = depths)
  lapply(seq(n), function(i) t(gl[seq(i, nrow(gl), n),]))
}

## simulate sequencing data based on the genotypes and mean sequencing depth a error rate
set.seed(1)
gl <- sim_gl(gt, mean_depth = 6, error = 0.01)

cat("Genotype likelihoods for individual 1")
gl[[1]]

 - How many sites are there in the output, and what do the three rows mean?
 - Which genotype is the most likely for SNP 1?
 - Find a site where all three genotype likelihoods are equal. What was the depth there, and
   what does that tell you about how much that site can contribute?

## Calling genotypes

The simplest thing to do with genotype likelihoods is to pick the genotype that makes the data
most likely

$$\widehat{G}_{ij}=\arg\max_g\; p(X_{ij}\mid G_{ij}=g)$$

This is a maximum likelihood genotype call with a flat prior.

In [ ]:
# Calls genotypes from genotype likelihoods with uniform prior.
call_gt <- function(gl) {
  gt <- do.call("rbind", lapply(gl, function(x) apply(x, 2, which.max) - 1))
  rownames(gt) <- paste("Ind",1:length(gl))
  colnames(gt) <- paste("SNP",1:length(gl[[1]][1,]))
  gt
}

gt_calls <- call_gt(gl)

cat("Genotypes call from GL (first 6 SNPs)")
gt_calls[,1:6]

 - Compare `gt_calls[,1:6]` with the true genotypes `gt[,1:6]`. How many are wrong?
 - A site covered by exactly one read can never be called a heterozygote. Why not? (look at
   the table of read probabilities above)

Now build the SFS from the called genotypes, exactly as we did for the true genotypes.

In [ ]:
call_sfs <- calc_sfs(gt_calls)
barplot(call_sfs,xlab="Number of derived alleles",ylab="fraction of sites",col=2,main="SFS from genotype calls")

 - Compare this SFS with the true one from before. Which categories are too large and which
   are too small?

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/sfs_quiz2.json")


# 3. Estimating the SFS from genotype likelihoods

Instead of calling genotypes we treat the allele count $Z_j$ as a **hidden variable** and
estimate the SFS by maximum likelihood. The SFS itself is the parameter: $\theta_z$ is the
probability that a random site has $z$ derived alleles, so it doubles as the prior on the
hidden state, $p(Z_j=z\mid\theta)=\theta_z$.

$$p(X\mid\theta)=\prod_{j=1}^m p(X_j\mid\theta)=\prod_{j=1}^m\sum_{z=0}^{2n}p(X_j\mid Z_j=z)\,p(Z_j=z\mid\theta)$$

This is the same shape as every other EM problem in this course: a sum over a hidden state,
with one factor that depends on the data and one that is the parameter. Estimating
$\hat\theta=\arg\max_\theta p(X\mid\theta)$ is done in two steps

 1. compute $p(X_j\mid Z_j=z)$ for every site and every $z$ - the **SAF**
 2. run an EM algorithm on those numbers

The nice thing is that step 1 only has to be done **once**: the SAF does not depend on
$\theta$, so it is computed up front and the EM then only touches a $(2n+1)\times m$ matrix.

## Step 1: the sample allele frequency likelihood (SAF)

$X_j=(X_{1j},\dots,X_{nj})$ is the sequencing data of all individuals at site $j$. We need the
probability of that data given that the sample carries exactly $z$ derived alleles

$$p(X_{j}|Z_j=z)=\sum_{g_{1j}}\sum_{g_{2j}}\ldots \sum_{g_{nj}}p(G_j=(g_{1j},\ldots,g_{nj})|Z_j=z)\prod_{i=1}^n p(X_{ij}|G_{ij}=g_{ij})$$

In words: sum over every combination of individual genotypes that adds up to $z$ derived
alleles, and for each combination multiply the genotype likelihoods together (the individuals
are independent given their genotypes).

Written like that the sum is hopeless - there are $3^n$ genotype combinations, which for
$n=10$ is already 59049 per site and for 100 individuals is more than the number of atoms in
the universe. The trick is to add the individuals **one at a time**: if you know the
distribution of the allele count over the first $i-1$ individuals, adding individual $i$ only
requires combining it with that individual's three genotype likelihoods. That turns $3^n$ into
something proportional to $n^2$, and it is what the loop in `calc_saf` below does.

In [ ]:
# Input is a list of 3-by-m matrices, output is a 2*n+1-by-m matrix.
calc_saf <- function(gl) {
  n <- length(gl)
  m <- unique(sapply(gl, ncol))
  saf <- matrix(0, nrow = 2 * n + 1, ncol = m)
  c <- choose(2, 0:2)
  saf[1:3, ] <- c * gl[[1]]
  for (i in 2:n) {
    # Loop invariant: after ith iteration, first 2 * i + 1 rows of SAF
    # would contain SAF for first i individuals if divided by
    # choose(2 * i, 0:(2 * i)))
        g <- gl[[i]]
        for (j in (2 * i - 1):1) {
            saf[j + 2,] <- colSums(saf[(j + 2):j,] * c * g)
        }
    saf[2,] <- saf[2,] * g[1,] + saf[1,] * 2 * g[2,]
    saf[1,] <- saf[1,] * g[1,]
  }
  saf / choose(2 * n, 0:(2 * n))
}


saf <- calc_saf(gl)

 - `saf` has one row per possible allele count and one column per site. What are its dimensions?
 - The function divides by `choose(2*n, 0:(2*n))` at the end. Which term of the formula above
   is that?

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/sfs_quiz3.json")


## Step 2: the EM algorithm

With the SAF in hand the EM algorithm is short. The hidden state is the allele count $Z_j$ and
the parameter is the SFS itself.

**E-step** - the posterior probability of each allele count at each site, by Bayes' formula

$$q(Z_{j}=z)=p(Z_j=z\mid X_j,\theta)=\frac{p(X_j|Z_j=z)\,p(Z_j=z|\theta)}{\sum_{z'=0}^{2n} p(X_j|Z_j=z')\,p(Z_j=z'|\theta)}
=\frac{\text{SAF}_{zj}\,\theta_z}{\sum_{z'}\text{SAF}_{z'j}\,\theta_{z'}}$$

**M-step** - the new SFS is the average posterior over all sites

$$\theta_{z}^{new}=\frac{\sum_{j=1}^m q(Z_{j}=z)}{\sum_{j=1}^m \sum_{z'=0}^{2n} q(Z_{j}=z')}=\frac{1}{m}\sum_{j=1}^m q(Z_j=z)$$

The denominator is just $m$, because the posterior of each site sums to one. In other words:
count each site as a *fraction* of a site in every category, instead of forcing it into the
single category of its called genotype. That is exactly what the genotype calling approach
could not do.

In [ ]:
# Run an E-step.
#
# Input is SFS and precalculated (2 * n + 1)-by-m SAF matrix,
# output is (2 * n + 1)-by-m matrix of posterior allele count probabilities.
e_step <- function(sfs, saf) {
  post <- sfs * saf
  sweep(post, 2, colSums(post), `/`)
}
# Run an M-step.
#
# Input is (2 * n + 1)-by-m matrix of posterior allele count probabilities,
# output is new SFS.
m_step <- function(post) {
  new <- rowSums(post)
  new / sum(new)
}
# Runs EM for fixed number of iterations.
em <- function(saf, iterations = 50) {

  sfs <- rep(1 / nrow(saf), nrow(saf))
  for (i in seq(iterations)) {
    sfs <- m_step(e_step(sfs, saf))
  }
  sfs
}


em_sfs <- em(saf, iterations = 50)
barplot(em_sfs,xlab="Number of derived alleles",ylab="fraction of sites",col=3,main="SFS from GL (EM algo)")

 - Find the E-step and the M-step in the code. Where is $\theta_z$ and where is the SAF?
 - The EM is started from a flat SFS, `rep(1/nrow(saf), nrow(saf))`. Why does the starting
   point matter less here than in the admixture model?
 - Try `em(saf, iterations = 1)` and `em(saf, iterations = 1000)`. How much does it change?

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/sfs_quiz4.json")


# 4. Comparing the three spectra

Now we can plot the truth, the EM estimate from genotype likelihoods and the estimate from
called genotypes next to each other.

In [ ]:
# Plot different SFS
all_sfs <- rbind(
"True" = true_sfs,
"EM" = em_sfs,
"Calls" = call_sfs
)
barplot(all_sfs,beside=T,col=1:3,
  legend=c("TRUE SFS","from GL (EM algorithm)","from called genotypes"),
  names=0:(2*n),ylab="fraction of sites",main="SFS ")

 - Which of the two estimates is closest to the truth?
 - Look at the rare end of the spectrum (1, 2 and 3 derived alleles) and at the category with
   0 derived alleles. Where does genotype calling go wrong, and in which direction?

## Experiments

Go back and change the simulation, rerun the cells and look at the plot again.

 - At which depth is the SFS from called genotypes no longer noticeably biased? Try
   `mean_depth = 1`, `2`, `4`, `10` and `30`.
 - What happens if you set the sequencing error rate to 0% or to 10%?
 - What happens if you increase the number of individuals `n` but keep the depth low? Does the
   EM estimate hold up better than the genotype calls?
 - What happens with fewer sites, e.g. `m = 1000`? Which of the two approaches gets noisy first?

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/em_algorithms/quiz/sfs_quiz5.json")


# Bonus

 - Write up the likelihood and the EM algorithm for estimating the **two dimensional** SFS of
   two populations. What is the hidden state now, and how many categories does the parameter
   have for $n_1$ and $n_2$ individuals?
 - The EM above gives every site the same weight. What would you do with sites that have no
   reads at all in any individual?
 - Real data has no way of knowing which allele is derived and which is ancestral unless an
   outgroup is used. If you do not know, you can only count the *minor* allele, which gives the
   **folded** SFS. How many categories does the folded SFS have?